In [0]:
%sql
USE CATALOG data_analyst_demo;
USE SCHEMA ecommerce;

In [0]:
customers = spark.table("silver_customers")
orders = spark.table("silver_orders")
order_items = spark.table("silver_order_items")
products = spark.table("silver_product")
payments = spark.table("silver_payments")
reviews = spark.table("silver_reviews")
sellers = spark.table("silver_sellers")
geolocation = spark.table("silver_geolocation")
category = spark.table("silver_category_translation")




In [0]:
orders = spark.table("data_analyst_demo.ecommerce.silver_orders")
customers = spark.table("data_analyst_demo.ecommerce.silver_customers")
order_items = spark.table("data_analyst_demo.ecommerce.silver_order_items")
products = spark.table("data_analyst_demo.ecommerce.silver_product")
payments = spark.table("data_analyst_demo.ecommerce.silver_payments")
sellers = spark.table("data_analyst_demo.ecommerce.silver_sellers")

In [0]:
orders = spark.table("silver_orders")
customers = spark.table("silver_customers")
order_items = spark.table("silver_order_items")
products = spark.table("silver_product")
payments = spark.table("silver_order_payments")
sellers = spark.table("silver_sellers")

In [0]:
%sql
SHOW TABLES IN data_analyst_demo.ecommerce;

database,tableName,isTemporary
ecommerce,bronze_category_translation,false
ecommerce,bronze_customers,false
ecommerce,bronze_geolocation,false
ecommerce,bronze_order_items,false
ecommerce,bronze_orders,false
ecommerce,bronze_payments,false
ecommerce,bronze_products,false
ecommerce,bronze_reviews,false
ecommerce,bronze_sellers,false
ecommerce,gold_monthly_sales,false


In [0]:
order_payments = spark.table("data_analyst_demo.ecommerce.silver_order_payments")
products = spark.table("data_analyst_demo.ecommerce.silver_product")

## 

## gold_sales

In [0]:
sales_df = (
    orders
    .join(customers, "customer_id", "left")
    .join(order_items, "order_id", "left")
    .join(products, "product_id", "left")
    .join(payments, "order_id", "left")
    .join(sellers, "seller_id", "left")
)

sales_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("data_analyst_demo.ecommerce.gold_sales")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7185128173437315>, line 13
      1 sales_df = (
      2     orders
      3     .join(customers, "customer_id", "left")
   (...)
      7     .join(sellers, "seller_id", "left")
      8 )
     10 sales_df.write \
     11     .format("delta") \
     12     .mode("overwrite") \
---> 13     .saveAsTable("data_analyst_demo.ecommerce.gold_sales")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:737, in DataFrameWriter.saveAsTable(self, name, format, mode, partitionBy, **options)
    735 self._write.table_name = name
    736 self._write.table_save_method = "save_as_table"
--> 737 _, _, ei = self._spark.client.execute_command(
    738     self._write.command(self._spark.client), self._write.observations
    739 )
    740 self._callback(ei)

File /databricks/python/lib/python3.12/site-packages/

## gold_product_summary

In [0]:
from pyspark.sql.functions import sum

product_summary = (
    sales_df
    .groupBy("product_id")
    .agg(
        sum("payment_value").alias("total_revenue")
    )
    .orderBy("total_revenue", ascending=False)
)
product_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("data_analyst_demo.ecommerce.gold_product_summary")

In [0]:
from pyspark.sql.functions import sum

gold_product_summary = (
    order_items_df
    .join(products_df, "product_id", "left")
    .groupBy("product_category_name")
    .agg(
        sum("price").alias("total_revenue")
    )
)
gold_product_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_product_summary")

In [0]:
display(spark.table("data_analyst_demo.ecommerce.gold_product_summary"))

product_category_name,total_revenue
ferramentas_jardim,485256.46000001475
papelaria,230943.22999999588
moveis_sala,68916.5600000002
musica,6034.3499999999985
livros_importados,4639.849999999998
moveis_colchao_e_estofado,4368.08
telefonia,323667.529999989
casa_conforto,58572.04000000023
fashion_underwear_e_moda_praia,9541.549999999987
informatica_acessorios,911954.3200000388


In [0]:
gold_product_summary.printSchema()

root
 |-- product_category_name: string (nullable = true)
 |-- total_revenue: double (nullable = true)



In [0]:
gold_product_summary = spark.table(
    "data_analyst_demo.ecommerce.gold_product_summary"
)

display(gold_product_summary)

product_category_name,total_revenue
ferramentas_jardim,485256.46000001475
papelaria,230943.22999999588
moveis_sala,68916.5600000002
musica,6034.3499999999985
livros_importados,4639.849999999998
moveis_colchao_e_estofado,4368.08
telefonia,323667.529999989
casa_conforto,58572.04000000023
fashion_underwear_e_moda_praia,9541.549999999987
informatica_acessorios,911954.3200000388


In [0]:
order_items_df = spark.table("data_analyst_demo.ecommerce.silver_order_items")

products_df = spark.table("data_analyst_demo.ecommerce.silver_product")

In [0]:
display(products_df.limit(5))

product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15
cef67bcfe19066a932b7673e239eb23d,bebes,27,261,1,371,26,4,26
9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37,402,4,625,20,17,13


In [0]:
%sql
drop table if exists gold_product_summary;

## gold_seller_summary

In [0]:
from pyspark.sql.functions import sum, desc

gold_seller_summary = (
    order_items_df
    .join(sellers_df, "seller_id", "left")
    .groupBy("seller_id")
    .agg(
        sum("price").alias("total_revenue")
    )
    .orderBy(desc("total_revenue"))
    .limit(10)
)

display(gold_seller_summary)

seller_id,total_revenue
4869f7a5dfa277a7dca6462dcf3b52b2,229472.6299999981
53243585a1d6dc2643021fd1853d8905,222776.04999999952
4a3ca9315b744ce9f8e9374361493884,200472.9199999949
fa1c13f2614d7b5c4749cbc52fecda94,194042.02999999846
7c67e1448b00f6e969d365cea6b010ab,187923.8899999995
7e93a43ef30c4f03f38b393420bc753a,176431.86999999982
da8622b14eb17ae2831f4ac5b9dab84a,160236.56999999538
7a67c85e85bb2ce8582c35f2203ad736,141745.53000000177
1025f0e2d44d7041d6cf58b6550e0bfa,138968.5499999995
955fee9216a65b617aa5c0531780ce60,135171.69999999914


In [0]:
gold_seller_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("data_analyst_demo.ecommerce.gold_seller_summary")

In [0]:
%sql
CREATE OR REPLACE TABLE gold_order_status_summary AS
SELECT
    order_status,
    COUNT(*) AS total_orders,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (),
        2
    ) AS percentage
FROM gold_sales
GROUP BY order_status;

num_affected_rows,num_inserted_rows


In [0]:
spark.sql("DROP TABLE IF EXISTS data_analyst_demo.ecommerce.gold_seller_summary")

DataFrame[]

## gold_state_summary

In [0]:
state_summary = (
    sales_df
    .groupBy("state")
    .agg(
        sum("payment_value").alias("state_revenue")
    )
    .orderBy("state_revenue", ascending=False)
)
state_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_state_summary")

## gold_payment_summary

In [0]:
from pyspark.sql.functions import count

payment_summary = (
    payments
    .groupBy("payment_type")
    .agg(
        count("*").alias("number_of_payments"),
        sum("payment_value").alias("total_amount")
    )
)
payment_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_payment_summary")

## gold_review_summary

In [0]:
from pyspark.sql.functions import avg

review_summary = (
    reviews
    .groupBy("review_score")
    .agg(
        count("*").alias("number_of_reviews")
    )
    .orderBy("review_score")
)
review_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_review_summary")

## gold_monthly_sales

In [0]:
from pyspark.sql.functions import month, year

monthly_sales = (
    sales_df
    .groupBy(
        year("order_purchase_timestamp").alias("year"),
        month("order_purchase_timestamp").alias("month")
    )
    .agg(
        sum("payment_value").alias("monthly_revenue")
    )
    .orderBy("year", "month")
)
monthly_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_monthly_sales")

In [0]:
%sql
show tables;

database,tableName,isTemporary
ecommerce,bronze_category_translation,false
ecommerce,bronze_customers,false
ecommerce,bronze_geolocation,false
ecommerce,bronze_order_items,false
ecommerce,bronze_orders,false
ecommerce,bronze_payments,false
ecommerce,bronze_products,false
ecommerce,bronze_reviews,false
ecommerce,bronze_sellers,false
ecommerce,gold_monthly_sales,false


In [0]:
%sql
show tables;

database,tableName,isTemporary
ecommerce,bronze_category_translation,false
ecommerce,bronze_customers,false
ecommerce,bronze_geolocation,false
ecommerce,bronze_order_items,false
ecommerce,bronze_orders,false
ecommerce,bronze_payments,false
ecommerce,bronze_products,false
ecommerce,bronze_reviews,false
ecommerce,bronze_sellers,false
ecommerce,gold_monthly_sales,false


In [0]:
%sql
DROP TABLE IF EXISTS data_analyst_demo.ecommerce.gold_product_summary;

In [0]:
from pyspark.sql.functions import sum

gold_product_summary = (
    order_items_df
    .join(products_df, "product_id", "left")
    .groupBy("product_category_name")
    .agg(
        sum("price").alias("total_revenue")
    )
)

gold_product_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("data_analyst_demo.ecommerce.gold_product_summary")

In [0]:
spark.table("data_analyst_demo.ecommerce.gold_product_summary").printSchema()

root
 |-- product_category_name: string (nullable = true)
 |-- total_revenue: double (nullable = true)



In [0]:
spark.sql("DROP TABLE IF EXISTS data_analyst_demo.ecommerce.gold_product_summary")

DataFrame[]

In [0]:
sellers_df = spark.table("data_analyst_demo.ecommerce.silver_sellers")

products_df = spark.table("data_analyst_demo.ecommerce.silver_product")

In [0]:
from pyspark.sql.functions import sum, desc

gold_product_summary = (
    order_items_df
    .join(products_df, "product_id", "left")
    .groupBy("product_category_name")
    .agg(
        sum("price").alias("total_revenue")
    )
    .orderBy(desc("total_revenue"))
    .limit(10)
)

display(gold_product_summary)

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5911811522748335>, line 4
      1 from pyspark.sql.functions import sum, desc
      3 gold_product_summary = (
----> 4     order_items_df
      5     .join(products_df, "product_id", "left")
      6     .groupBy("product_category_name")
      7     .agg(
      8         sum("price").alias("total_revenue")
      9     )
     10     .orderBy(desc("total_revenue"))
     11     .limit(10)
     12 )
     14 display(gold_product_summary)

NameError: name 'order_items_df' is not defined

In [0]:
gold_product_summary.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("data_analyst_demo.ecommerce.gold_product_summary")

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-5911811522748337>, line 1
----> 1 gold_product_summary.write \
      2     .format("delta") \
      3     .mode("overwrite") \
      4     .saveAsTable("data_analyst_demo.ecommerce.gold_product_summary")

NameError: name 'gold_product_summary' is not defined

## 1.Total_Revenue

In [0]:
%sql
SELECT
ROUND(SUM(payment_value)/1000000, 2) AS Total_Revenue_Million
FROM gold_sales;

Total_Revenue_Million
20.47


## 2,Total_orders


In [0]:
%sql
SELECT
COUNT(DISTINCT order_id) AS Total_Orders
FROM gold_sales;

Total_Orders
99441


## 3,Total_Customers

In [0]:
%sql
SELECT
COUNT(DISTINCT customer_unique_id) AS Total_Customers
FROM gold_sales;

Total_Customers
96096


## 4,Average order value

In [0]:
%sql
SELECT
ROUND(
SUM(payment_value) /
COUNT(DISTINCT order_id),
2
) AS Avg_Order_Value
FROM gold_sales;

Avg_Order_Value
205.86


## 5, Total sallers

In [0]:
%sql
SELECT
COUNT(DISTINCT seller_id) AS Total_Sellers
FROM gold_sales;

Total_Sellers
3095


In [0]:
%sql
SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
data_analyst_demo,ecommerce


In [0]:
%sql
SHOW TABLE EXTENDED IN data_analyst_demo.ecommerce LIKE 'gold_sales';

database,tableName,isTemporary,information
ecommerce,gold_sales,false,"Catalog: data_analyst_demo Database: ecommerce Table: gold_sales Created Time: Fri Jul 31 16:48:39 UTC 2026 Last Access: UNKNOWN Created By: Spark Type: MANAGED Provider: delta Table Properties: [delta.enableDeletionVectors=true, delta.parquet.compression.codec=zstd, delta.parquet.format.version=2.12.0, delta.parquet.format.version.afe.internal=2.12.0] Statistics: 12751006 bytes, 118434 rows Location: s3://dbstorage-prod-h9pt9z2p1n/uc/7bcb7227-211a-4609-819b-b5e5079c45ff/e630980b-85d1-47c6-8eeb-e5940055d7a9/__unitystorage/catalogs/fc911509-ff9d-4090-b7b6-bbc9c643a123/tables/f406f1a6-7317-4610-bc85-b0d596b6b3d0 Partition Provider: Catalog Schema: root |-- seller_id: string (nullable = true) |-- order_id: string (nullable = true) |-- product_id: string (nullable = true) |-- customer_id: string (nullable = true) |-- order_status: string (nullable = true) |-- order_purchase_timestamp: timestamp (nullable = true) |-- order_approved_at: timestamp (nullable = true) |-- order_delivered_carrier_date: timestamp (nullable = true) |-- order_delivered_customer_date: timestamp (nullable = true) |-- order_estimated_delivery_date: timestamp (nullable = true) |-- customer_unique_id: string (nullable = true) |-- zip_code: integer (nullable = true) |-- city: string (nullable = true) |-- state: string (nullable = true) |-- order_item_id: integer (nullable = true) |-- shipping_limit_date: timestamp (nullable = true) |-- price: double (nullable = true) |-- freight_value: double (nullable = true) |-- product_category_name: string (nullable = true) |-- product_name_lenght: integer (nullable = true) |-- product_description_lenght: integer (nullable = true) |-- product_photos_qty: integer (nullable = true) |-- product_weight_g: integer (nullable = true) |-- product_length_cm: integer (nullable = true) |-- product_height_cm: integer (nullable = true) |-- product_width_cm: integer (nullable = true) |-- payment_sequential: integer (nullable = true) |-- payment_type: string (nullable = true) |-- payment_installments: integer (nullable = true) |-- payment_value: double (nullable = true) |-- seller_zip_code_prefix: integer (nullable = true) |-- seller_city: string (nullable = true) |-- seller_state: string (nullable = true) Predictive Optimization: ENABLE (inherited from METASTORE metastore_aws_us_east_2)"
